# Copyright Rayyan Hodges, TAFE NSW, Gelos Enterprises, Indigo Community Services and Health Hub © 2026

# Contact: rayyan.hodges@studytafensw.edu.au, rayyan.hodges@gelosmail.com.au indigoCSHH@hmail.com

# Program Name: HopsitalCancerClassification.ipynb

# Purpose: To train and evaluate an ML model to automate the task of classifying clients as high, medium or low risk of developing cancer.

In [16]:
# Import required libraries
import joblib
import numpy
import pandas as pd
import sklearn # importing the scikit-learn project, different name due to syntax rules 
import scipy
import threadpoolctl

In [17]:
# Set appropriate display options in Pandas
pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 10000)
pd.set_option('display.width', 10000)

# Identify any datapoints with missing labels and remove them

In [18]:
# Load the data set into memory and define it for further analysis.
df = pd.read_csv("data.csv")
# Check for missing values in the target column, count them, and display them in a total number.
df['cancer_risk'].isnull().sum()

np.int64(10)

In [4]:
# Remove the affected rows with missing data
df = df.dropna(subset=['cancer_risk'])
# Verify removal of said invalid data.
df['cancer_risk'].isnull().sum()

np.int64(0)

# Categorise ages as required.

In [19]:
# Define age bins and labels for the groups
# Bins meaning boundaries 
bins = [0, 14, 24, 34, 44, 54, 64, 74, 84, 120]
labels = ['0-14', '15-24', '25-34', '35-44', 
          '45-54', '55-64', '65-74', '75-84', '85+']

# Categorise ages into age groups
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

# Display the results in a table
df[['age', 'age_group']].head()


,age,age_group
0,51,45-54
1,92,85+
2,14,0-14
3,71,65-74
4,60,55-64


# Standardise numerical features using appropriate scaling techniques.

In [20]:
# Import the scaling component from the SciKit Learning Library
from sklearn.preprocessing import StandardScaler

# Select numerical features from the datasheet.
numerical_features = ['bmi', 'blood_pressure']

# Initialise scaler component
scaler = StandardScaler()

# Apply scaling and transformation to standardise the numbers.
df[numerical_features] = scaler.fit_transform(df[numerical_features])

# Display results from the scaling and transformation.
df[numerical_features].describe()


,bmi,blood_pressure
count,1.000000e+03,1.000000e+03
mean,1.456613e-16,5.879741e-16
std,1.000500e+00,1.000500e+00
min,-1.684915e+00,-1.703831e+00
25%,-8.885765e-01,-8.621512e-01
50%,3.871853e-03,-1.077505e-02
75%,8.860227e-01,8.687217e-01
max,1.733849e+00,1.775369e+00


# Encode categorical data using appropriate encoding techniques.

In [23]:
# Identify categorical features from the dataset file to be referenced.
categorical_features = ['age_group', 'smoking_history', 'diet', 'exercise_frequency']

# Apply one-hot encoding to convert the text categories into dummy numerical categories (specifically true or false)
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

# Print the results to ensure the categorical features have been converted to variables such as true or false correctly.
df_encoded.head()


,age,family_history,alcohol_consumption,bmi,blood_pressure,cholesterol_level,travelled_overseas,number_of_children,cancer_risk,age_group_15-24,age_group_25-34,age_group_35-44,age_group_45-54,age_group_55-64,age_group_65-74,age_group_75-84,age_group_85+,smoking_history_former,smoking_history_never,diet_good,diet_poor,exercise_frequency_low,exercise_frequency_moderate,exercise_frequency_none
0,51,no,low,0.484421,1.631857,226.4,no,5,moderate,False,False,False,True,False,False,False,False,False,True,False,True,False,False,True
1,92,no,moderate,-1.492695,1.399135,170.9,yes,2,low,False,False,False,False,False,False,False,True,False,True,False,True,False,False,False
2,14,no,none,0.525611,0.569092,279.9,yes,4,high,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
3,71,yes,low,1.665199,-0.101925,280.7,yes,0,moderate,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False
4,60,no,none,-1.451505,0.650544,212.8,yes,5,moderate,False,False,False,False,True,False,False,False,True,False,False,False,False,False,True


# Divide and randomise the dataset into training

In [25]:

# Remove rows with missing target labels (specifically, the 'cancer_risk' label, as it has missing data) 
df_encoded = df_encoded.dropna(subset=['cancer_risk'])

#Import the splitting model from the scikit-learn model.
from sklearn.model_selection import train_test_split

# Separate features and target
X = df_encoded.drop('cancer_risk', axis=1)
y = df_encoded['cancer_risk']

# First split: 70% training, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Print the results of said sizes to ensure it is working correctly.
X_train.shape, X_val.shape, X_test.shape


((693, 23), (148, 23), (149, 23))

# Select the most important features in the training dataset that may influence cancer risk, dropping 2 unnecessary features. - Rayyan Hodges

In [26]:

# Drop two unnecessary features/columns that aren't relevant to cancer risk evaluation and wouldn't affect if someone would be likely to have cancer.
X_train = X_train.drop(['travelled_overseas', 'number_of_children'], axis=1)
X_val   = X_val.drop(['travelled_overseas', 'number_of_children'], axis=1)
X_test  = X_test.drop(['travelled_overseas', 'number_of_children'], axis=1)

# Print the results to allow the user to determine the changes have worked
X_train.shape


(693, 21)